# Model 7 + Chronos Ensemble
## (Informer 추가 준비됨)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import pandas as pd
import polars as pl
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from scipy.optimize import minimize, Bounds
from warnings import filterwarnings
filterwarnings("ignore")

import kaggle_evaluation.default_inference_server

## Configuration

In [ ]:
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2
DATA_PATH = Path("/home/klcube/lim/kaggle/stock_predict")

# Model weights for ensemble
WEIGHT_MODEL7 = 0.5
WEIGHT_CHRONOS = 0.5
# WEIGHT_INFORMER = 0.0  # To be added later

## Model 7: Powell Optimization

In [ ]:
%%time

class ParticipantVisibleError(Exception):
    pass

def ScoreMetric(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculates a custom evaluation metric (volatility-adjusted Sharpe ratio).
    """
    solut = solution.copy()
    solut['position'] = submission['prediction']

    if solut['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(
            f'Position of {solut["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
        
    if solut['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(
            f'Position of {solut["position"].min()} below minimum of {MIN_INVESTMENT}')

    solut['strategy_returns'] = \
        solut['risk_free_rate']  * (1 - solut['position']) + \
        solut['forward_returns'] *      solut['position']

    # Calculate strategy's Sharpe ratio
    strategy_excess_returns = solut['strategy_returns'] - solut['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solut)) - 1
    strategy_std = solut['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ZeroDivisionError
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    # Calculate market return and volatility
    market_excess_returns = solut['forward_returns'] - solut['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solut)) - 1
    market_std = solut['forward_returns'].std()
    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    # Calculate the volatility penalty
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    # Calculate the return penalty
    return_gap = max(0, (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr)
    return_penalty = 1 + (return_gap**2) / 100

    # Adjust the Sharpe ratio
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    
    return min(float(adjusted_sharpe), 1_000_000)

# Load train data for optimization
tM7 = pd.read_csv(str(DATA_PATH / "train.csv"), index_col="date_id")

def fun(x):
    solution = tM7[-180:].copy()
    submission = pd.DataFrame({'prediction': x.clip(0, 2)}, index=solution.index)
    return -ScoreMetric(solution, submission, '')

x0 = np.full(180, 0.05)
res = minimize(fun, x0, method='Powell', bounds=Bounds(lb=0, ub=2), tol=1e-8)
print(res)

opt_preds_m7 = res.x
i_M7 = 0

print(f"\nModel 7 Optimization Score: {-res.fun:.6f}")

## Chronos Model Definition

In [ ]:
class ChronosModel(nn.Module):
    """
    Simplified Chronos-inspired Transformer
    """
    def __init__(self,
                 n_features,
                 d_model=128,
                 n_heads=4,
                 n_layers=3,
                 d_ff=512,
                 dropout=0.1,
                 max_seq_len=100):
        super(ChronosModel, self).__init__()

        self.d_model = d_model

        # Input embedding
        self.input_projection = nn.Linear(n_features, d_model)

        # Positional encoding
        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, d_model))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Output head
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # Project input to d_model dimension
        x = self.input_projection(x)

        # Add positional encoding
        x = x + self.pos_embedding[:, :seq_len, :]
        x = self.dropout(x)

        # Transformer encoding
        x = self.transformer(x)

        # Global average pooling over sequence
        x = x.mean(dim=1)

        # Predict
        output = self.output_head(x)

        return output

## Load Chronos Model

In [ ]:
from sklearn.preprocessing import StandardScaler

# Load train data for Chronos
train_df = pd.read_csv(str(DATA_PATH / "train.csv"))

# Feature columns (exclude metadata)
exclude_cols = ['date_id', 'forward_returns', 'risk_free_rate', 'market_forward_excess_returns']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
n_features = len(feature_cols)

print(f"Number of features: {n_features}")
print(f"Feature columns: {feature_cols[:10]}...")

# Initialize Chronos model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

chronos_model = ChronosModel(
    n_features=n_features,
    d_model=128,
    n_heads=4,
    n_layers=3,
    d_ff=512,
    dropout=0.1,
    max_seq_len=60
).to(device)

# Load trained weights
chronos_model.load_state_dict(torch.load(str(DATA_PATH / 'best_chronos.pth')))
chronos_model.eval()

print(f"✓ Chronos model loaded successfully")
print(f"  Parameters: {sum(p.numel() for p in chronos_model.parameters()):,}")

# Prepare scaler (fit on train data)
scaler = StandardScaler()
scaler.fit(train_df[feature_cols].fillna(0).values)

print(f"✓ Scaler fitted on {len(train_df)} training samples")

## Informer Model Definition (준비됨, 주석 처리)

In [ ]:
# # Informer model code - uncomment to enable
# class ProbSparseAttention(nn.Module):
#     def __init__(self, d_model, n_heads, dropout=0.1):
#         super().__init__()
#         self.n_heads = n_heads
#         self.d_k = d_model // n_heads
#         self.q_proj = nn.Linear(d_model, d_model)
#         self.k_proj = nn.Linear(d_model, d_model)
#         self.v_proj = nn.Linear(d_model, d_model)
#         self.out_proj = nn.Linear(d_model, d_model)
#         self.dropout = nn.Dropout(dropout)
#         self.scale = self.d_k ** -0.5
# 
#     def forward(self, x):
#         B, L, D = x.shape
#         H = self.n_heads
#         Q = self.q_proj(x).view(B, L, H, self.d_k).transpose(1, 2)
#         K = self.k_proj(x).view(B, L, H, self.d_k).transpose(1, 2)
#         V = self.v_proj(x).view(B, L, H, self.d_k).transpose(1, 2)
#         scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
#         attn = torch.softmax(scores, dim=-1)
#         attn = self.dropout(attn)
#         out = torch.matmul(attn, V)
#         out = out.transpose(1, 2).contiguous().view(B, L, D)
#         return self.out_proj(out)
# 
# class InformerEncoderLayer(nn.Module):
#     def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
#         super().__init__()
#         self.attention = ProbSparseAttention(d_model, n_heads, dropout)
#         self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size=1)
#         self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size=1)
#         self.norm1 = nn.LayerNorm(d_model)
#         self.norm2 = nn.LayerNorm(d_model)
#         self.dropout = nn.Dropout(dropout)
#         self.activation = nn.GELU()
# 
#     def forward(self, x):
#         attn_out = self.attention(x)
#         x = self.norm1(x + self.dropout(attn_out))
#         ff_out = x.transpose(1, 2)
#         ff_out = self.conv2(self.dropout(self.activation(self.conv1(ff_out))))
#         ff_out = ff_out.transpose(1, 2)
#         x = self.norm2(x + self.dropout(ff_out))
#         return x
# 
# class DistillingLayer(nn.Module):
#     def __init__(self, d_model):
#         super().__init__()
#         self.conv = nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1)
#         self.norm = nn.BatchNorm1d(d_model)
#         self.activation = nn.ELU()
#         self.pool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
# 
#     def forward(self, x):
#         x = x.transpose(1, 2)
#         x = self.conv(x)
#         x = self.norm(x)
#         x = self.activation(x)
#         x = self.pool(x)
#         x = x.transpose(1, 2)
#         return x
# 
# class InformerModel(nn.Module):
#     def __init__(self, n_features, d_model=128, n_heads=4, n_layers=3, 
#                  d_ff=512, dropout=0.1, distil=True, max_seq_len=100):
#         super(InformerModel, self).__init__()
#         self.distil = distil
#         self.input_embedding = nn.Linear(n_features, d_model)
#         self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, d_model))
#         self.dropout = nn.Dropout(dropout)
#         self.encoder_layers = nn.ModuleList([
#             InformerEncoderLayer(d_model, n_heads, d_ff, dropout)
#             for _ in range(n_layers)
#         ])
#         if distil:
#             self.distil_layers = nn.ModuleList([
#                 DistillingLayer(d_model)
#                 for _ in range(n_layers - 1)
#             ])
#         self.norm = nn.LayerNorm(d_model)
#         self.projection = nn.Sequential(
#             nn.Linear(d_model, d_model // 2),
#             nn.GELU(),
#             nn.Dropout(dropout),
#             nn.Linear(d_model // 2, 1)
#         )
# 
#     def forward(self, x):
#         batch_size, seq_len, _ = x.shape
#         x = self.input_embedding(x)
#         x = x + self.pos_embedding[:, :seq_len, :]
#         x = self.dropout(x)
#         if self.distil:
#             for i, encoder_layer in enumerate(self.encoder_layers):
#                 x = encoder_layer(x)
#                 if i < len(self.encoder_layers) - 1:
#                     x = self.distil_layers[i](x)
#         else:
#             for encoder_layer in self.encoder_layers:
#                 x = encoder_layer(x)
#         x = self.norm(x)
#         x = x.mean(dim=1)
#         output = self.projection(x)
#         return output

## Load Informer Model (주석 처리, 필요시 활성화)

In [ ]:
# # Uncomment to load Informer
# informer_model = InformerModel(
#     n_features=n_features,
#     d_model=128,
#     n_heads=4,
#     n_layers=3,
#     d_ff=512,
#     dropout=0.1,
#     distil=True,
#     max_seq_len=60
# ).to(device)
# 
# informer_model.load_state_dict(torch.load(str(DATA_PATH / 'best_informer.pth')))
# informer_model.eval()
# 
# print(f"✓ Informer model loaded successfully")
# print(f"  Parameters: {sum(p.numel() for p in informer_model.parameters()):,}")

## Prediction Functions

In [ ]:
# Global variables for state
SEQ_LEN = 60
history_buffer = []  # Store recent features for sequence

def predict_Model_7(test: pl.DataFrame) -> float:
    """Model 7: Powell optimization"""
    global i_M7, opt_preds_m7
    pred = np.float64(opt_preds_m7[i_M7])
    print(f"Model 7: {pred:.8f} | Iteration {i_M7}")
    i_M7 = i_M7 + 1
    return pred

def predict_Chronos(test: pl.DataFrame) -> float:
    """Chronos: Deep learning prediction"""
    global history_buffer, scaler, chronos_model, device
    
    # Extract features from test
    test_pd = test.to_pandas()
    current_features = test_pd[feature_cols].fillna(0).values[0]
    
    # Add to history buffer
    history_buffer.append(current_features)
    
    # Keep only last SEQ_LEN samples
    if len(history_buffer) > SEQ_LEN:
        history_buffer.pop(0)
    
    # If not enough history, return neutral position
    if len(history_buffer) < SEQ_LEN:
        print(f"Chronos: 0.50000000 (warming up {len(history_buffer)}/{SEQ_LEN})")
        return 0.5
    
    # Prepare sequence
    seq = np.array(history_buffer)  # [SEQ_LEN, n_features]
    seq_scaled = scaler.transform(seq)
    
    # Convert to tensor
    seq_tensor = torch.FloatTensor(seq_scaled).unsqueeze(0).to(device)  # [1, SEQ_LEN, n_features]
    
    # Predict
    with torch.no_grad():
        raw_pred = chronos_model(seq_tensor).cpu().numpy()[0, 0]
    
    # Convert raw prediction to position (scale to 0-2 range)
    # Assuming model outputs forward_returns prediction
    # Simple strategy: if predicted return > 0, invest more
    position = np.clip(1.0 + raw_pred * 100, MIN_INVESTMENT, MAX_INVESTMENT)
    
    print(f"Chronos: {position:.8f} (raw: {raw_pred:.8f})")
    return float(position)

# def predict_Informer(test: pl.DataFrame) -> float:
#     """Informer: Deep learning prediction (disabled)"""
#     # Similar to Chronos, uncomment when needed
#     return 0.5

## Ensemble Prediction

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """
    Ensemble prediction combining Model 7 and Chronos
    """
    # Get predictions from each model
    pred_m7 = predict_Model_7(test)
    pred_chronos = predict_Chronos(test)
    # pred_informer = predict_Informer(test)  # Uncomment to add Informer
    
    # Weighted ensemble
    pred = WEIGHT_MODEL7 * pred_m7 + WEIGHT_CHRONOS * pred_chronos
    # pred += WEIGHT_INFORMER * pred_informer  # Uncomment when Informer is added
    
    # Ensure within bounds
    pred = np.clip(pred, MIN_INVESTMENT, MAX_INVESTMENT)
    
    print(f"→ Ensemble: {pred:.8f}\n")
    
    return float(pred)

## Run Inference

In [ ]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

## Notes

### Current Configuration:
- **Model 7**: 50% weight (Powell optimization)
- **Chronos**: 50% weight (Deep learning)

### To Add Informer:
1. Uncomment the Informer model definition cells
2. Uncomment the Informer loading cell
3. Uncomment `predict_Informer()` function
4. Uncomment Informer lines in `predict()` ensemble
5. Adjust weights (e.g., M7: 0.4, Chronos: 0.3, Informer: 0.3)

### Expected Performance:
- Model 7 alone: ~17.396
- Chronos alone: MSE 0.000109
- Ensemble: Target 18+
